# HopSkipJump: Tree vs SVM under Coverage-Gap Bias (iris)

**Question:** does within-cluster density of adversarial points track dataset bias, and does the signal transfer from Decision Tree to SVM when both face the *same* black-box attack?

**Pipeline per run:** inject bias -> 5-fold CV -> HopSkipJump on correctly-classified test points -> OPTICS clustering -> **within-cluster density = n_points / (mean pairwise distance + 1)** (Aiden's metric, with the FIXED distance norm `||p1 - p2||`).

**Resumable:** results checkpoint to `results.parquet` every 18 runs; re-running the notebook skips completed runs. Progress is also written to `progress.txt`.

In [2]:
# Limit BLAS/OpenMP threads BEFORE importing numpy - prevents oversubscription
import os
for _v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[_v] = '1'

import numpy as np, polars as pl, time, warnings, os
from pathlib import Path
from itertools import product
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import OPTICS
from scipy.spatial.distance import pdist
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump
warnings.filterwarnings('ignore')

HERE = Path.cwd()
OUT = HERE / 'results.parquet'
PROGRESS = HERE / 'progress.txt'
print('Output folder:', HERE)

Output folder: e:\OneDrive - The University of Auckland\Desktop\Model Auditing\Auditing\hsj_svm_experiment


## Config

In [3]:
BIAS_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
SEEDS       = [42, 58, 125]
CLASSES     = [0, 1, 2]
FEATURES    = [0, 1, 2, 3]
N_FOLDS     = 5
CHECKPOINT_EVERY = 6

models = {
    'tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'svm':  SVC(kernel='rbf', probability=True, random_state=42),
}

grid = list(product(BIAS_LEVELS, SEEDS, CLASSES, FEATURES, models.items()))
TOTAL = len(grid)
print(f'{len(BIAS_LEVELS)} bias x {len(SEEDS)} seeds x {len(CLASSES)} classes x {len(FEATURES)} features x {len(models)} models = {TOTAL} runs')
print(f'Estimated: ~{TOTAL*3/60:.0f} min')

5 bias x 3 seeds x 3 classes x 4 features x 2 models = 360 runs
Estimated: ~18 min


## Functions

`cluster_density` is the key metric: for each OPTICS cluster, `n_points / (mean pairwise distance + 1)`.
Higher = more points packed more tightly. This is Aiden's original formula with the distance bug fixed.

In [ ]:
def inject_bias(X, y, target_class, feature_idx, bias):
    """Coverage gap: delete the bottom `bias` fraction of target_class sorted by feature."""
    if bias == 0:
        return X.copy(), y.copy()
    mask = y == target_class
    order = np.argsort(X[mask][:, feature_idx])
    Xs, ys = X[mask][order], y[mask][order]
    n_keep = max(int(len(Xs) * (1 - bias)), 3)
    return (np.vstack([X[~mask], Xs[-n_keep:]]),
            np.hstack([y[~mask], ys[-n_keep:]]))


def cluster_density(points, min_samples=3):
    """Within-cluster density (Aiden's metric, fixed norm).
    For each OPTICS cluster: density = n_points / (mean pairwise dist + 1).
    Returns (mean density across clusters, n_clusters)."""
    if len(points) < min_samples + 1:
        return np.nan, np.nan
    o = OPTICS(min_samples=min_samples, xi=0.05, min_cluster_size=min_samples).fit(points)
    densities = []
    for c in set(o.labels_) - {-1}:
        pts = points[o.labels_ == c]
        if len(pts) < 2:
            continue
        mean_dist = pdist(pts).mean()   # mean pairwise Euclidean distance, CORRECT norm
        densities.append(len(pts) / (mean_dist + 1))
    if not densities:
        return np.nan, 0
    return float(np.mean(densities)), len(densities)


def gen_adv(art_model, X, y):
    """HopSkipJump on correctly-classified points. Returns (successful adv points, success rate)."""
    p = np.argmax(art_model.predict(X), axis=1)
    c = p == y
    Xc, yc = X[c], y[c]
    if len(Xc) == 0:
        return np.array([]), 0.0
    hs = HopSkipJump(classifier=art_model, norm=2, max_iter=10, max_eval=200,
                     init_eval=50, verbose=False)
    adv = hs.generate(Xc)
    ap = np.argmax(art_model.predict(adv), axis=1)
    ok = ap != yc
    return adv[ok], float(ok.mean())


def run_one(X, y, model_sklearn, target_class, feature_idx, bias, seed):
    np.random.seed(seed)
    Xb, yb = inject_bias(X, y, target_class, feature_idx, bias)
    ohe = OneHotEncoder(sparse_output=False)
    yoh = ohe.fit_transform(yb.reshape(-1, 1))
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    folds = []
    for tr, te in skf.split(Xb, yb):
        Xt, Xv, yt, yv = Xb[tr], Xb[te], yb[tr], yb[te]
        m = model_sklearn.__class__(**model_sklearn.get_params()).fit(Xt, yt)
        art = SklearnClassifier(m)
        art.fit(Xt, yoh[tr])
        adv, succ = gen_adv(art, Xv, yv)
        d, nc = cluster_density(adv)
        folds.append({'tacc': m.score(Xt, yt), 'vacc': m.score(Xv, yv),
                       'asucc': succ, 'nadv': len(adv), 'density': d, 'nclust': nc})
    r = {k: float(np.nanmean([f[k] for f in folds])) for k in folds[0]}
    r.update(seed=seed, bias=round(bias, 2), tc=target_class, feat=feature_idx)
    return r

: 

## Main Loop (resumable, checkpoints every 18 runs)

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

# Resume support: load existing results, skip completed runs
if OUT.exists():
    done_df = pl.read_parquet(OUT)
    done_keys = set(zip(done_df['bias'].round(2), done_df['seed'], done_df['tc'],
                         done_df['feat'], done_df['model']))
    all_rows = done_df.to_dicts()
    print(f'Resuming: {len(all_rows)} runs already done')
else:
    done_keys = set()
    all_rows = []

todo = [(b, s, c, f, nm, m) for (b, s, c, f, (nm, m)) in grid
        if (round(b, 2), s, c, f, nm) not in done_keys]
print(f'{len(todo)} runs to go')

t0 = time.time()
for j, (bias, seed, tc, feat, nm, m) in enumerate(todo):
    r = run_one(X, y, m, tc, feat, bias, seed)
    r['model'] = nm
    all_rows.append(r)

    if (j + 1) % CHECKPOINT_EVERY == 0 or j + 1 == len(todo):
        elapsed = time.time() - t0
        pct = (j + 1) / len(todo)
        eta = elapsed / pct - elapsed
        msg = f'[{j+1}/{len(todo)}] {pct:.0%}  elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m'
        print(msg, flush=True)
        pl.DataFrame(all_rows).write_parquet(OUT)
        PROGRESS.write_text(f'{msg}\ntotal_rows={len(all_rows)}/{TOTAL}\n')

print(f'\nDONE: {len(all_rows)} total rows in {(time.time()-t0)/60:.1f} min')
df = pl.DataFrame(all_rows)
df.head()

Resuming: 6 runs already done
354 runs to go
[6/354] 2%  elapsed=0.3m  ETA=16.2m
[12/354] 3%  elapsed=0.6m  ETA=16.0m
[18/354] 5%  elapsed=0.9m  ETA=15.9m
[24/354] 7%  elapsed=1.1m  ETA=15.7m
[30/354] 8%  elapsed=1.4m  ETA=15.4m
[36/354] 10%  elapsed=1.7m  ETA=14.8m
[42/354] 12%  elapsed=2.0m  ETA=14.6m
[48/354] 14%  elapsed=2.3m  ETA=14.4m
[54/354] 15%  elapsed=2.5m  ETA=13.9m
[60/354] 17%  elapsed=2.8m  ETA=13.6m
[66/354] 19%  elapsed=3.1m  ETA=13.5m
[72/354] 20%  elapsed=3.4m  ETA=13.2m
[78/354] 22%  elapsed=3.6m  ETA=12.9m
[84/354] 24%  elapsed=3.9m  ETA=12.6m


## Results Table

In [ ]:
df = pl.read_parquet(OUT)
agg = df.group_by(['model', 'bias']).agg(
    pl.col('density').mean().alias('density'),
    pl.col('density').std().alias('d_std'),
    pl.col('nclust').mean().alias('clusters'),
    pl.col('vacc').mean().alias('test_acc'),
    pl.col('asucc').mean().alias('adv_rate'),
    pl.col('nadv').mean().alias('n_adv'),
).sort(['model', 'bias'])
agg

## Plot: Within-Cluster Density (z-scored, 95% CI) + Accuracy

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

df_z = df.with_columns(
    ((pl.col('density') - pl.col('density').mean()) / pl.col('density').std())
    .over('model').alias('dz')
)
az = df_z.group_by(['model', 'bias']).agg(
    pl.col('dz').mean().alias('z'), pl.col('dz').std().alias('zs'),
    pl.col('dz').count().alias('n'), pl.col('vacc').mean().alias('test_acc'),
).sort(['model', 'bias'])
az = az.with_columns(
    pl.struct(['z','zs','n']).map_elements(
        lambda r: r['z'] - stats.t.ppf(0.975, r['n']-1)*r['zs']/np.sqrt(r['n']),
        return_dtype=pl.Float64).alias('lo'),
    pl.struct(['z','zs','n']).map_elements(
        lambda r: r['z'] + stats.t.ppf(0.975, r['n']-1)*r['zs']/np.sqrt(r['n']),
        return_dtype=pl.Float64).alias('hi'),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, nm in zip(axes, ['tree', 'svm']):
    d = az.filter(pl.col('model') == nm).sort('bias')
    x = d['bias'].to_numpy()
    ax.plot(x, d['test_acc'], 's-', color='#1f77b4', lw=2, ms=6, label='Test Acc')
    ax.set_ylabel('Accuracy', color='#1f77b4'); ax.tick_params(axis='y', labelcolor='#1f77b4')
    ax.set_ylim(0.6, 1.05)
    ax2 = ax.twinx()
    c = '#2ca02c' if nm == 'tree' else '#d62728'
    ax2.fill_between(x, d['lo'], d['hi'], color=c, alpha=0.12)
    ax2.plot(x, d['z'], 'o--', color=c, lw=2, ms=6, label='Within-cluster density (z)')
    ax2.axhline(0, color='gray', ls=':', lw=1)
    ax2.set_ylabel('Density (z)', color=c); ax2.tick_params(axis='y', labelcolor=c)
    ax.set_xlabel('Bias level'); ax.set_title(nm.upper(), fontweight='bold'); ax.set_xlim(0.05, 0.95)
    h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, loc='lower left', fontsize=9)
fig.suptitle('HopSkipJump: Within-Cluster Density vs Coverage Gap (iris)', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('plots/density_vs_bias.png', dpi=200, bbox_inches='tight')
plt.show()

## Statistical Test (bias 0.1 vs 0.9)

In [ ]:
for nm in ['tree', 'svm']:
    lo = df.filter((pl.col('model')==nm) & (pl.col('bias')==0.1))['density'].to_numpy()
    hi = df.filter((pl.col('model')==nm) & (pl.col('bias')==0.9))['density'].to_numpy()
    lo, hi = lo[~np.isnan(lo)], hi[~np.isnan(hi)]
    t, p = stats.ttest_ind(lo, hi)
    d = (hi.mean() - lo.mean()) / np.sqrt((lo.var() + hi.var()) / 2)
    print(f'{nm:>6s}: density {lo.mean():.4f} -> {hi.mean():.4f}   t={t:.2f}  p={p:.4f}  cohens_d={d:.3f}')

---
**Metric:** within-cluster density = `n_points / (mean pairwise distance + 1)` per OPTICS cluster, averaged across clusters. Fixed norm (`pdist`), unlike the original buggy `np.linalg.norm((p1, p2))`.\n
**Attack:** HopSkipJump `max_iter=10, max_eval=200, init_eval=50`, L2, untargeted - identical for both models.\n
**Resume:** just re-run the notebook; completed runs are skipped via `results.parquet`.